# Nerual network model for regret prediction with cross validation

## Import and Random seed

In [1]:
import copy
import random
import numpy as np
import pandas as pd
import ast
import re

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import ray
from ray import tune
from ray.tune.schedulers import ASHAScheduler
from ray.tune.search.optuna import OptunaSearch

In [2]:
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

## 2. Data Processing

In [3]:
def parse_ks_result(value):
    if pd.isna(value):
        return np.nan, np.nan
    value = str(value)
    stat_match = re.search(
        r"statistic\s*=\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)",
        value
    )
    pvalue_match = re.search(
        r"pvalue\s*=\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)",
        value
    )
    ks_stat = float(stat_match.group(1)) if stat_match else np.nan
    ks_pvalue = float(pvalue_match.group(1)) if pvalue_match else np.nan
    return ks_stat, ks_pvalue

def parse_pred_res(value):
    if isinstance(value, dict):
        return value
    if pd.isna(value):
        return {}
    try:
        result = ast.literal_eval(value)
        return result if isinstance(result, dict) else {}
    except (ValueError, SyntaxError):
        return {}

In [4]:
# Read data from a CSV file into a pandas DataFrame
df = pd.read_csv("dataset/tpc-ds/regret_queries_100_add9scores_scaled.csv")

df[["ks_stat", "ks_pvalue"]] = df["ks_result"].apply(
    lambda value: pd.Series(parse_ks_result(value))
)
df["pred_res_dict"] = df["pred_res"].apply(parse_pred_res)
pred_columns = ["query_id", "chosen_idx", "optimal_idx", "bao_latency", "optimal_latency", "default_latency",
                "regret", ]
for column in pred_columns:
    df[column] = df["pred_res_dict"].apply(
        lambda result: result.get(column, np.nan)
    )

In [5]:
def extract_metric(text, metric_name):
    if not isinstance(text, str):
        return np.nan

    # Locate the metric pattern (e.g., mmd=0.3060)
    pattern = rf"{metric_name}=([\d.]+)"
    match = re.search(pattern, text)
    return float(match.group(1)) if match else np.nan

def extract_array(text, array_name):
    if not isinstance(text, str):
        return None

    # Locate the array structure inside the string
    pattern = rf"{array_name}=array\(\[(.*?)\]\)"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        # Clean up inner text spacing and newlines
        clean_arr_text = match.group(1).replace('\\n', '').replace('\n', '')
        # Split by comma or whitespace and filter out empty strings
        items = [float(x) for x in re.split(r'[\s,]+', clean_arr_text) if x.strip()]
        return np.array(items)
    return None

# 1. Extract single scalar metrics directly into new columns
df['mmd'] = df['new_result'].apply(lambda x: extract_metric(x, 'mmd'))
df['energy_distance'] = df['new_result'].apply(lambda x: extract_metric(x, 'energy_distance'))
df['wasserstein'] = df['new_result'].apply(lambda x: extract_metric(x, 'wasserstein'))
df['ks'] = df['new_result'].apply(lambda x: extract_metric(x, 'ks'))
df['js_divergence'] = df['new_result'].apply(lambda x: extract_metric(x, 'js_divergence'))
df['js_distance'] = df['new_result'].apply(lambda x: extract_metric(x, 'js_distance'))
df['kl_x_to_y'] = df['new_result'].apply(lambda x: extract_metric(x, 'kl_x_to_y'))
df['kl_y_to_x'] = df['new_result'].apply(lambda x: extract_metric(x, 'kl_y_to_x'))
df['symmetric_kl'] = df['new_result'].apply(lambda x: extract_metric(x, 'symmetric_kl'))

# 2. Extract the nested featurewise arrays back into true NumPy arrays
df['featurewise_wasserstein'] = df['new_result'].apply(lambda x: extract_array(x, 'featurewise_wasserstein'))
df['featurewise_ks'] = df['new_result'].apply(lambda x: extract_array(x, 'featurewise_ks'))
df['featurewise_js'] = df['new_result'].apply(lambda x: extract_array(x, 'featurewise_js'))
df['featurewise_kl_x_to_y'] = df['new_result'].apply(lambda x: extract_array(x, 'featurewise_kl_x_to_y'))

# Test output to verify it works perfectly:
print("First row MMD:", df['mmd'].iloc[0])
print("First row Wasserstein:", df['wasserstein'].iloc[0])
print("First row Featurewise KS Array Shape:", df['featurewise_ks'].iloc[0].shape)

First row MMD: 1.0858129543928186
First row Wasserstein: 1.2087855884499397
First row Featurewise KS Array Shape: (64,)


In [6]:
df.columns

Index(['mmd_score', 'ks_result', 'ws_score', 'pred_res', 'new_result',
       'ks_stat', 'ks_pvalue', 'pred_res_dict', 'query_id', 'chosen_idx',
       'optimal_idx', 'bao_latency', 'optimal_latency', 'default_latency',
       'regret', 'mmd', 'energy_distance', 'wasserstein', 'ks',
       'js_divergence', 'js_distance', 'kl_x_to_y', 'kl_y_to_x',
       'symmetric_kl', 'featurewise_wasserstein', 'featurewise_ks',
       'featurewise_js', 'featurewise_kl_x_to_y'],
      dtype='object')

## Retrieve all need columns

In [7]:
final_df = df[["query_id","mmd_score", "ks_stat", "ks_pvalue", "ws_score", "bao_latency", "optimal_latency", "default_latency","regret","mmd","energy_distance","wasserstein","ks","js_divergence","js_distance","kl_x_to_y","kl_y_to_x","symmetric_kl","featurewise_wasserstein","featurewise_ks","featurewise_js","featurewise_kl_x_to_y"]].copy()
# 确保全部是数值类型
# for column in final_df.columns:
#     final_df[column] = pd.to_numeric(final_df[column], errors="coerce")
#
# # 删除缺失值和无穷值
# final_df = final_df.replace([np.inf, -np.inf], np.nan)
# final_df = final_df.dropna().reset_index(drop=True)

In [8]:
final_df["ks_log_pvalue"] = -np.log10(final_df["ks_pvalue"].clip(lower=1e-300))
final_df["reg_log"] = np.log10(final_df["regret"].clip(lower=1e-300))

In [9]:
final_df.to_pickle("dataset/tpc-ds/precessed_regret_queries_100_add9scores_scaled.pkl")